# Component 3 — CRMA Bayesian Network Risk Inference

C3 combines exceedance probabilities from the Zarr store (or inline evidence when the
store is still being built) with admin-1 boundaries to drive the Compound Risk Model
for the IGAD region (CRMA). Each admin-1 unit receives a four-state risk label
(Green/Yellow/Orange/Red) with marginal probabilities that sum to 1.

**Key classes**: `CRMAModel`, `CRMAEvidence`

In [ ]:
import os, subprocess, sys
from pathlib import Path
from datetime import date

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt


def _bootstrap() -> Path:
    """Locate the repo; on Colab, clone + install it first."""
    for p in (Path.cwd(), *Path.cwd().parents):
        if (p / "configs" / "default.yaml").exists():
            return p
    repo = Path.cwd() / "gik-icechain"
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/hashirama21/gik-icechain.git", str(repo)], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"{repo}[dev]"], check=True)
    return repo


REPO = _bootstrap()
DATA = REPO / "data"

# Download every prerequisite from public sources (idempotent): admin
# boundaries, CMORPH return periods, ENSO/IOD index, then threshold files.
tools = [sys.executable, str(REPO / "scripts" / "tools.py")]
subprocess.run([*tools, "download", "--component", "all"], check=True)
subprocess.run([*tools, "download-thresholds"], check=True)

from gik_icechain.shared.config import load_config

# Optional .env credentials (live MinIO store; offline mode works without)
_env = REPO / ".env"
if _env.exists():
    for _line in _env.read_text().splitlines():
        if _line and not _line.startswith("#") and "=" in _line:
            _k, _v = _line.split("=", 1)
            os.environ.setdefault(_k.strip(), _v.strip())
os.environ.setdefault("AWS_ACCESS_KEY_ID", os.environ.get("MINIO_ACCESS_KEY", ""))
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", os.environ.get("MINIO_SECRET_KEY", ""))

cfg = load_config(REPO / "configs" / "default.yaml")
STORAGE_OPTIONS = {"endpoint_url": cfg.outputs.endpoint_url}

START = date(2025, 1, 1)
END   = date(2025, 1, 2)
print(f"Repo: {REPO}")
print(f"Config loaded — endpoint: {cfg.outputs.endpoint_url}")
print(f"Demo window: {START} → {END}")

## 3.1 Build CRMA model

In [ ]:
from gik_icechain.risk.crma_model import CRMAModel, CRMAEvidence

model = CRMAModel(crma_cfg=cfg.component3.crma_model)
model.build()
print("CRMA model built.")
print("Nodes:", list(model._model.nodes()))

## 3.2 Load exceedance store

In [ ]:
try:
    exc_ds = xr.open_zarr(
        cfg.outputs.exceedance_store_uri,
        consolidated=False,
        storage_options=STORAGE_OPTIONS,
    )
    print("Exceedance store available:")
    print(exc_ds)
except Exception as exc:
    exc_ds = None
    print("C2 store not available — using inline evidence for demo")
    print(f"({type(exc).__name__}: {exc})")

## 3.3 Load admin boundaries

In [ ]:
import geopandas as gpd

admin = gpd.read_file(DATA / "admin_boundaries" / "east_africa_admin1.geojson")
print(f"{len(admin)} admin-1 units")
print(admin[["admin1_pcode", "shapeName", "geometry"]].head(5))

## 3.4 Single-unit inference from exceedance store

In [ ]:
if exc_ds is not None:
    target_date = pd.Timestamp(START)
    date_coord  = "date" if "date" in exc_ds.coords else "time"
    unit        = admin.iloc[0]
    bounds      = unit.geometry.bounds  # (minx, miny, maxx, maxy)
    lat_dim     = cfg.component2.spatial.lat_dim
    lon_dim     = cfg.component2.spatial.lon_dim
    exc_day     = exc_ds.sel({date_coord: target_date}, method="nearest")
    exc_unit    = exc_day.sel(
        {lat_dim: slice(bounds[1], bounds[3]),
         lon_dim: slice(bounds[0], bounds[2])},
    )
    p24  = float(exc_unit["exceedance_prob"].sel(window=24, return_period=5).mean())
    p72  = float(exc_unit["exceedance_prob"].sel(window=72, return_period=5).mean())
    p168 = float(exc_unit["exceedance_prob"].sel(window=168, return_period=5).mean())
    cov  = float((exc_unit["exceedance_prob"].sel(window=24, return_period=5) > 0.15).mean())
    evidence = CRMAEvidence(
        exceedance_prob_24h=p24,
        exceedance_prob_72h=p72,
        exceedance_prob_7d=p168,
        gpm_obs_24h=0.0,
        api_mm=20.0,
        spatial_coverage_fraction=cov,
        consecutive_signal_days=1,
        sat_consecutive_days=0,
    )
else:
    evidence = CRMAEvidence(
        exceedance_prob_24h=0.35,
        exceedance_prob_72h=0.25,
        exceedance_prob_7d=0.18,
        gpm_obs_24h=0.0,
        api_mm=20.0,
        spatial_coverage_fraction=0.3,
        consecutive_signal_days=1,
        sat_consecutive_days=0,
    )

result = model.infer(evidence)
print(f"Risk label : {result['risk_label']}")
print(f"Green={result['p_green']:.3f}  Yellow={result['p_yellow']:.3f}  "
      f"Orange={result['p_orange']:.3f}  Red={result['p_red']:.3f}")

## 3.5 Batch inference across all units (from exceedance store or example values)

In [ ]:
records = []

if exc_ds is not None:
    date_coord = "date" if "date" in exc_ds.coords else "time"
    exc_day    = exc_ds.sel({date_coord: pd.Timestamp(START)}, method="nearest")
    lat_dim    = cfg.component2.spatial.lat_dim
    lon_dim    = cfg.component2.spatial.lon_dim
    for _, unit in admin.iloc[:10].iterrows():
        bounds   = unit.geometry.bounds
        exc_unit = exc_day.sel(
            {lat_dim: slice(bounds[1], bounds[3]),
             lon_dim: slice(bounds[0], bounds[2])},
        )
        if exc_unit.sizes.get(lat_dim, 0) == 0 or exc_unit.sizes.get(lon_dim, 0) == 0:
            continue
        p24  = float(exc_unit["exceedance_prob"].sel(window=24, return_period=5).mean())
        p72  = float(exc_unit["exceedance_prob"].sel(window=72, return_period=5).mean())
        p168 = float(exc_unit["exceedance_prob"].sel(window=168, return_period=5).mean())
        cov  = float((exc_unit["exceedance_prob"].sel(window=24, return_period=5) > 0.15).mean())
        ev   = CRMAEvidence(
            exceedance_prob_24h=p24, exceedance_prob_72h=p72,
            exceedance_prob_7d=p168, gpm_obs_24h=0.0, api_mm=20.0,
            spatial_coverage_fraction=cov, consecutive_signal_days=1,
            sat_consecutive_days=0,
        )
        res = model.infer(ev)
        records.append({"admin1_pcode": unit["admin1_pcode"],
                        "admin1_name": unit["shapeName"],
                        "risk_label": res["risk_label"],
                        "risk_state": res["risk_state"],
                        "p_green": res["p_green"], "p_yellow": res["p_yellow"],
                        "p_orange": res["p_orange"], "p_red": res["p_red"]})
else:
    example_scenarios = [
        {"label": "KE-Nairobi",  "p24": 0.05, "p72": 0.03, "p168": 0.02, "cov": 0.05},
        {"label": "ET-Oromia",   "p24": 0.18, "p72": 0.12, "p168": 0.08, "cov": 0.25},
        {"label": "UG-Central",  "p24": 0.32, "p72": 0.25, "p168": 0.18, "cov": 0.40},
        {"label": "SO-Shabelle", "p24": 0.52, "p72": 0.45, "p168": 0.38, "cov": 0.65},
        {"label": "TZ-Kilimanj", "p24": 0.10, "p72": 0.07, "p168": 0.05, "cov": 0.12},
    ]
    for sc in example_scenarios:
        ev  = CRMAEvidence(
            exceedance_prob_24h=sc["p24"], exceedance_prob_72h=sc["p72"],
            exceedance_prob_7d=sc["p168"], gpm_obs_24h=0.0, api_mm=20.0,
            spatial_coverage_fraction=sc["cov"], consecutive_signal_days=1,
            sat_consecutive_days=0,
        )
        res = model.infer(ev)
        records.append({"admin1_pcode": sc["label"], "admin1_name": sc["label"],
                        "risk_label": res["risk_label"], "risk_state": res["risk_state"],
                        "p_green": res["p_green"], "p_yellow": res["p_yellow"],
                        "p_orange": res["p_orange"], "p_red": res["p_red"]})

risk_df = pd.DataFrame(records).sort_values("risk_state", ascending=False)
print(risk_df.to_string(index=False))

## 3.6 Risk distribution visualization

In [ ]:
state_colors = {"Green": "#2ecc71", "Yellow": "#f1c40f", "Orange": "#e67e22", "Red": "#e74c3c"}
counts = risk_df["risk_label"].value_counts().reindex(["Green", "Yellow", "Orange", "Red"], fill_value=0)

if exc_ds is not None and len(records) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    ax.bar(counts.index, counts.values,
           color=[state_colors[s] for s in counts.index])
    ax.set_title("Risk label distribution")
    ax.set_ylabel("Admin-1 units")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.05, str(v), ha="center")

    merged = admin.merge(risk_df[["admin1_pcode", "risk_label"]], on="admin1_pcode", how="left")
    merged["color"] = merged["risk_label"].map(state_colors).fillna("#cccccc")
    merged.plot(ax=axes[1], color=merged["color"], edgecolor="white", linewidth=0.3)
    axes[1].set_title("Risk map — East Africa")
    axes[1].set_xlabel("Longitude")
    axes[1].set_ylabel("Latitude")
else:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(counts.index, counts.values,
           color=[state_colors[s] for s in counts.index])
    ax.set_title("Risk label distribution (example scenarios)")
    ax.set_ylabel("Scenarios")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 0.05, str(v), ha="center")

fig.tight_layout()
plt.show()